# 06 — Fresh SPY-core overlay backtest and final results

Run the NEW notebook **05** to completion first. This notebook then replays **all 12** fixed model/scheme/threshold combinations in **both** validation and test, computes dividend-reinvestment SPY, generates new metrics, ranks the candidates and renders new figures. It does not read `spy_overlay_*_20261101` or historical images.

The original final presentation rule is preserved: choose the highest **test** net CAGR. This is an **exploratory ex-post selection**, not independent holdout evidence. Results are calculated dynamically; GBDT/S1/15 bp is not forced to win.


In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Image, Markdown
import pandas as pd

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "requirements.txt").is_file()
             and (p / "notebooks").is_dir() and (p / "scripts").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the project directory or a subdirectory.")
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
import fresh_backtest_pipeline as pipeline
print("Project:", ROOT)
print("Environment:", pipeline.runtime())


## 1. Connect to this project's completed fresh 05 run

`PAIR_RUN_DIR=None` uses the latest completed NEW run. Set an explicit path under `results/reproduction/` if desired. Inputs, engines and all completed 05 artifacts are hash-verified; changed predictions require a new 05 run. Missing fresh output raises an error rather than falling back to historical results.


In [ ]:
PAIR_RUN_DIR = None
ctx = pipeline.connect_pair_run(ROOT, PAIR_RUN_DIR)
OVERLAY_DIR = pipeline.start_overlay_run(ctx)
print("Parent 05 run:", ctx["out"])
print("Fresh 06 output:", OVERLAY_DIR)


## 2. Fresh validation replay: 12 candidates and reinvestment SPY

SPY is the core holding. Existing pair signals adjust sector weights; net exposure stays 100%, gross at most 150%. Preserve the existing fees, borrow and funding rules. Thresholds are 10/5 and 15/7.5 bp, with two model families and three feature schemes. Independently initialize every candidate.


In [ ]:
validation_metrics = pipeline.run_overlay_phase(ctx, "validation")
display(validation_metrics[["series_id", "annualized_net_return", "max_drawdown", "total_cost"]])


## 3. Fresh test replay: the same 12 fixed candidates

No new model fitting or threshold values. Recompute all cash/share ledgers, trades, costs and the reinvestment SPY benchmark. Runtime checks verify complete dates, final liquidation, return products, gross caps, fees, financing and accounting.


In [ ]:
test_metrics = pipeline.run_overlay_phase(ctx, "test")
display(test_metrics[["series_id", "annualized_net_return", "max_drawdown", "total_cost"]])


## 4. Compute ranking, selected comparison, yearly returns and figures

Rank the freshly computed test CAGRs with a fixed series-ID tie rule. Retain every candidate and both phases. Selected wealth is compared with fresh reinvestment SPY and the current 05 run's fresh fixed-share SPY/cash benchmarks. Calendar-year rows use actual intervals; 2026 ends on September 30 and is not annualized as a full year.

Relative performance, annualized tracking error and information ratio explicitly use **dividend-reinvestment SPY**, with exported names ending in `_vs_spy_reinvest`. Historical fixed-share SPY metrics have a different benchmark.


In [ ]:
summary, ranked, yearly = pipeline.finish_overlay_run(ctx)
display(ranked[["test_return_rank", "series_id", "annualized_net_return",
                "validation_annualized_net_return", "selected_for_final_presentation"]])


In [ ]:
fields = ["split", "series_id", "annualized_net_return", "annualized_net_volatility",
          "max_drawdown", "cumulative_relative_to_spy_reinvest",
          "annualized_tracking_error_vs_spy_reinvest", "information_ratio_vs_spy_reinvest"]
display(summary[fields])
for phase in pipeline.PHASE_OPENS:
    display(Image(filename=str(OVERLAY_DIR / f"selected_{phase}.png")))


In [ ]:
display(yearly)
selected_id = ranked.iloc[0].series_id
fields = ["split", "series_id", "sector_participation", "mean_holding_intervals", "mean_gross_exposure",
          "max_gross_exposure", "annualized_two_sided_turnover", "total_execution_fee",
          "total_borrow_fee", "total_financing_interest", "total_cost"]
display(summary.loc[summary.series_id.eq(selected_id), fields])
print("Complete:", OVERLAY_DIR)
print("All results above were generated by this run; selection is exploratory and ex post.")
